# M5: q_N/q_V 명시적 M2 + CLV 양성가중 M4 (test-only seed 42)

q_N은 M2 경제표현의 강도만 조절하고, q_V와 축소 4구간 지출분포가 상품 경제적 방향을 결정합니다. q_C는 M2에 중복 입력하지 않고 M4의 양성 학습우선순위에만 사용합니다. ID와 보조표현은 layer-0에서 결합되어 하나의 optimizer로 2층 전파·공동학습됩니다.

DAY 1~697을 학습하고 DAY 698~704 test를 고정 100 epoch 마지막 checkpoint에서 한 번만 평가합니다. 이 test 구간은 이미 노출됐으므로 결과는 탐색적 기술값이며, 결과를 보고 같은 test에 모형을 다시 맞추지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import importlib, os, shutil, subprocess, sys

REVIEWED_SHA = '281ded1c499f157115b28a995871bd514bae29cb'
REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
os.chdir('/content')
repo = Path('/content/clv-m2-lightgcn-runner')
clone_errors = []
for clone_attempt in range(1, 4):
    if repo.exists():
        shutil.rmtree(repo)
    result = subprocess.run(
        ['git', 'clone', REPO_URL, str(repo)],
        text=True, capture_output=True,
    )
    if result.returncode == 0:
        break
    clone_errors.append(result.stderr.strip())
    print(f'GitHub clone {clone_attempt}/3 실패:', result.stderr.strip())
else:
    raise RuntimeError('GitHub clone 3회 실패:\n' + '\n'.join(clone_errors))
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(
    ['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True
).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
for module_name in tuple(sys.modules):
    if module_name.startswith(('lightgcn_', 'clv_')):
        del sys.modules[module_name]
importlib.invalidate_caches()
%cd /content/clv-m2-lightgcn-runner
print('실행 코드 고정 완료:', actual_sha)

In [ ]:
import json
import torch
from lightgcn_clv_m5_nv_economic_positive_weight_test import (
    configure_m5_nv_economic_positive_test_run,
    preflight_summary,
    run_m5_nv_economic_positive_test,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_m5_nv_economic_positive_test_run(
    out_dir='/content/drive/MyDrive/논문/data/results_v3_dunnhumby_m5_explicit_nv_economic_positive_weighting_test_seed42_v1',
)
summary = preflight_summary(cfg)
assert cfg.seeds == (42,)
assert summary['validation_constructed'] is False
assert summary['holdout_evaluation'] is False
assert summary['m2']['q_c_used_in_m2'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
result_df = run_m5_nv_economic_positive_test(cfg)

In [ ]:
from IPython.display import display

print('1) seed 42 test 절대지표')
display(result_df)
print('2) M1·M2·M4·M5·순열 대조군 비교')
display(result_df.attrs['comparison'])
print("3) M2×M4 상호작용 — 보조 지표")
display(result_df.attrs['interaction'])
print('4) 사전 고정 판독')
print(json.dumps(result_df.attrs['descriptive_reading'], ensure_ascii=False, indent=2))
print('5) 저장 파일')
print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))